In [ ]:
pip install ipykernel pandas


In [ ]:
import pandas as pd
filename = "/Users/zienxuang/sg-food-mcp/data/raw/food_db.csv"
df = pd.read_csv(filename, na_values=["-", "", "NA"])
df.shape

In [ ]:
# how empty is a column?
df["Alternative Serving Size(s)"].notna().sum()

# what categories exist
df["Category"].value_counts()

# look at one dish
df[df["Food Name"].str.contains("kopi", case=False, na=False)][
    ["Food Name", "Default Serving Size", "Energy (kcal) - Per Serving"]
]

In [ ]:
import pandas as pd
import re

RAW = filename
OUT = "/Users/zienxuang/sg-food-mcp/data/dishes_draft.csv"

COLS = {
    "Food Name":                      "dish_name",
    "Description":                    "description",
    "Category":                       "category",
    "Food Group":                     "food_group",
    "Food Subgroup":                  "food_subgroup",
    "Default Serving Size":           "serving_default_desc",
    "Per Serving Size":               "serving_basis",
    "Per 100 Unit":                   "per100_unit",
    "Energy (kcal) - Per 100":        "kcal_per_100",
    "Energy (kcal) - Per Serving":    "kcal_default",
    "Protein (g) - Per Serving":      "protein_g",
    "Carbohydrate (g) - Per Serving": "carbs_g",
    "Total Fat (g) - Per Serving":    "fat_g",
    "Source of Data":                 "source",
    "Last Updated":                   "last_updated",
}

NUMERIC = ["kcal_per_100", "kcal_default", "protein_g", "carbs_g", "fat_g"]

def make_id(name):
    return re.sub(r"[^a-z0-9]+", "-", str(name).lower()).strip("-")[:60]

# '-' is this dataset's null marker
df = pd.read_csv(RAW, na_values=["-", "", "NA"])
df = df[list(COLS)].rename(columns=COLS)

for c in NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce")

before = len(df)
df = df[df["kcal_default"].notna()]
print(f"dropped {before - len(df)} rows with no per-serving energy")

df["id"] = df["dish_name"].map(make_id)
dupes = df["id"].duplicated().sum()
if dupes:
    print(f"warning: {dupes} duplicate ids — check these")
    print(df[df["id"].duplicated(keep=False)][["dish_name"]].to_string())
df = df.drop_duplicates(subset="id")

# beverages go to the tap path, not the camera
df["input_mode"] = df["category"].eq("Beverages").map({True: "tap", False: "photo"})

df["other_names"] = ""
df["portion_note"] = ""
df["is_favourite"] = False

order = ["id", "dish_name", "other_names", "category", "food_group", "food_subgroup",
         "description", "serving_default_desc", "serving_basis", "kcal_default",
         "kcal_per_100", "per100_unit",
         "protein_g", "carbs_g", "fat_g",
         "portion_note", "input_mode", "is_favourite", "source", "last_updated"]
df[order].to_csv(OUT, index=False)

print(f"\n{len(df)} rows -> {OUT}")
print(df["input_mode"].value_counts())
print(df["category"].value_counts().head(15))

In [ ]:
import pandas as pd
d = pd.read_csv('/Users/zienxuang/sg-food-mcp/data/raw/dishes.csv')
assert d['id'].is_unique, 'duplicate ids'
assert d['kcal_default'].notna().all(), 'missing kcal'
assert d['kcal_default'].between(0, 3500).all(), 'implausible kcal'
print(len(d), 'rows')
print(d['input_mode'].value_counts().to_dict())
print('aliases:', (d['other_names'].fillna('') != '').sum())
print('portion_note:', (d['portion_note'].fillna('') != '').sum())
print('favourites:', d['is_favourite'].sum())